In [ ]:
import torch

if torch.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

device

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
        "json",
        split="train",
        data_files="data/python-edu/val.jsonl",
    )["text"]


print(dataset[:1])

In [ ]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bigcode/starcoder2-3b")

class CoderDataset(Dataset):
    def __init__(self, dataset, tokenizer, context_length):
        all_tokens = []
        
        if tokenizer.eos_token_id is None:
                raise ValueError("Tokenizer must define eos_token_id.")

        for start in range(0, len(dataset), 100):
            texts = dataset[start : start + 100]
            encoded_batch = tokenizer(texts)["input_ids"]

            for encoded in encoded_batch:
                all_tokens.extend(encoded)
                all_tokens.append(tokenizer.eos_token_id)

        self.tokens = torch.tensor(all_tokens, dtype=torch.long) 
        self.context_length = context_length

    def __len__(self):
        return (len(self.tokens) - 1) // self.context_length

    def __getitem__(self, index):
        start = index * self.context_length
        chunk = self.tokens[start : start + self.context_length + 1]
        return chunk[:-1], chunk[1:]

In [ ]:
coder_dataset = CoderDataset(dataset, tokenizer, 512)
len(coder_dataset)
x, y = coder_dataset[0]
print(x.shape)
print(y.shape)
print(torch.all(x[1:] == y[:-1]))

In [ ]:
from torch.utils.data import DataLoader

data_loader = DataLoader(
    coder_dataset,
    8,
    shuffle=True,
    num_workers=0
)

x_batch, y_batch = next(iter(data_loader))
print(x_batch.shape)
print(y_batch.shape)